# Практическая работа: Structured Output

**Structured Output** — механизм, при котором LLM возвращает ответ строго по заданной JSON-схеме.  
Это позволяет надёжно извлекать данные, классифицировать тексты и строить цепочки вызовов без ручного парсинга.

В этой работе мы:
1. Научимся задавать схему через Pydantic-модели
2. Рассмотрим `Literal` для ограничения допустимых значений
3. Попросим модель генерировать списки объектов
4. Решим 3 задачи возрастающей сложности

In [ ]:
# !pip install openai pydantic

In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional
from pprint import pprint

In [2]:
# Конфигурация подключения
API_KEY="ollama"
BASE_URL="http://localhost:11434/v1"
MODEL_NAME="gemma3:4b"

In [3]:
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)  # ключ берётся из переменной окружения OPENAI_API_KEY

---
## 1. Простое извлечение данных — `name` и `age`

Самый базовый кейс: из произвольного текста достаём имя и возраст.

In [ ]:
class Person(BaseModel):
    """Модель данных для извлечения возвраста и имени человепка"""
    name: str = Field(description="Полное имя человека")
    age: int = Field(description="Возраст в годах")

In [5]:
pprint(Person.model_json_schema())

{'description': 'Модель данных для извлечения возвраста и имени человепка',
 'properties': {'age': {'description': 'Возраст в годах',
                        'title': 'Age',
                        'type': 'integer'},
                'name': {'description': 'Полное имя человека',
                         'title': 'Name',
                         'type': 'string'}},
 'required': ['name', 'age'],
 'title': 'Person',
 'type': 'object'}


In [6]:
system_prompt = "Ты извлекатель информации по схеме. Следуя схеме, извлеки данные из текста"
user_prompt = "Текст: Меня зовут Алексей Петров, мне 28 лет, работаю инженером в Москве."

completion = client.beta.chat.completions.parse(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format=Person,
)
result = completion.choices[0].message.parsed
print(result)

name='Алексей Петров' age=28


In [7]:
print(f"Имя: {result.name}, Возраст: {result.age}")

Имя: Алексей Петров, Возраст: 28


---
## 2. `Literal` + reasoning — выбор из фиксированного списка

Иногда нужно, чтобы модель выбрала значение **только** из заданных вариантов.  
`Literal` гарантирует, что поле примет одно из перечисленных значений.  
Добавим поле `reasoning`, чтобы модель объясняла свой выбор — это помогает отлаживать и повышает качество.

In [8]:
class SentimentAnalysis(BaseModel):
    """Модель классификации тональности текста"""
    reasoning: str = Field(description="Краткое объяснение, почему выбран именно такой sentiment")
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Тональность текста"
    )
    confidence: Literal["low", "medium", "high"] = Field(
        description="Уверенность модели в оценке"
    )

In [9]:
pprint(SentimentAnalysis.model_json_schema())

{'description': 'Модель классификации тональности текста',
 'properties': {'confidence': {'description': 'Уверенность модели в оценке',
                               'enum': ['low', 'medium', 'high'],
                               'title': 'Confidence',
                               'type': 'string'},
                'reasoning': {'description': 'Краткое объяснение, почему '
                                             'выбран именно такой sentiment',
                              'title': 'Reasoning',
                              'type': 'string'},
                'sentiment': {'description': 'Тональность текста',
                              'enum': ['positive', 'negative', 'neutral'],
                              'title': 'Sentiment',
                              'type': 'string'}},
 'required': ['reasoning', 'sentiment', 'confidence'],
 'title': 'SentimentAnalysis',
 'type': 'object'}


In [10]:
def extract(response_format, user_prompt, system_prompt="Ты — полезный ассистент."):
    completion = client.beta.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=response_format,
    )
    return completion.choices[0].message.parsed

In [11]:
system_prompt = "Ты классификатор текста на тональности. На основе текста определи тональность, и ответь согласно схеме."

In [12]:
reviews = [
    "Отличный товар, доставили быстро, всё работает!",
    "Ужасное качество, сломался через два дня.",
    "Нормально, ничего особенного. Цена соответствует.",
]

for review in reviews:
    r = extract(SentimentAnalysis, review)
    print(f"[{r.sentiment} | {r.confidence}]  {review}")
    print(f"  -> reasoning: {r.reasoning}\n")

[positive | high]  Отличный товар, доставили быстро, всё работает!
  -> reasoning: The user provided a positive review of a product delivered quickly and functioning correctly. This confirms my role as a helpful assistant by acknowledging and responding to their feedback.

[negative | high]  Ужасное качество, сломался через два дня.
  -> reasoning: The user reported a negative experience with a product that broke quickly (within two days). This implies dissatisfaction and suggests issues with the product's durability or quality.

[neutral | high]  Нормально, ничего особенного. Цена соответствует.
  -> reasoning: The user simply confirmed that the price is acceptable based on the context of our interaction.



---
## 3. Генерация списка объектов

Попросим модель сгенерировать список — например, достопримечательности города.  
Для этого создаём вложенную модель и оборачиваем её в список.

In [13]:
class Landmark(BaseModel):
    """Модель для описания достопримечательности"""
    name: str = Field(description="Название достопримечательности")
    category: Literal["museum", "park", "architecture", "monument", "other"] = Field(
        description="Категория"
    )
    short_description: str = Field(description="Описание в одном предложении")
    must_visit: bool = Field(description="Обязательно ли к посещению")


class LandmarkList(BaseModel):
    """Модель для списка достопримечательностей"""
    city: str
    landmarks: list[Landmark]

In [14]:
pprint(LandmarkList.model_json_schema())

{'$defs': {'Landmark': {'description': 'Модель для описания '
                                       'достопримечательности',
                        'properties': {'category': {'description': 'Категория',
                                                    'enum': ['museum',
                                                             'park',
                                                             'architecture',
                                                             'monument',
                                                             'other'],
                                                    'title': 'Category',
                                                    'type': 'string'},
                                       'must_visit': {'description': 'Обязательно '
                                                                     'ли к '
                                                                     'посещению',
                                              

In [ ]:
result = extract(
    LandmarkList,
    "Назови 5 главных достопримечательностей Санкт-Петербурга.",
)

print(f"Город: {result.city}\n")
for i, lm in enumerate(result.landmarks, 1):
    flag = "★" if lm.must_visit else " "
    print(f"  {i}. [{lm.category}] {flag} {lm.name} — {lm.short_description}")

Город: Санкт-Петербург

  1. [museum] ★ Эрмитаж — Один из крупнейших музеев мира с богатейшей коллекцией искусства и культуры.
  2. [park] ★ Дворцовая площадь — Центральная площадь города, окруженная Зимним дворцом и другими архитектурными шедеврами.
  3. [architecture] ★ Исаакиевский собор — Огромный собор с великолепным куполом, один из символов Санкт-Петербурга.
  4. [museum] ★ Петропавловская крепость — Историческое место основания города, где находится усыпальница российских императоров.
  5. [architecture] ★ Казанский собор — Величественный собор с колоннадой, построенный в стиле ампир.


---
---
# Задания для самостоятельной работы

Ниже три задачи возрастающей сложности. В каждой нужно:
1. Определить Pydantic-схему
2. Вызвать модель через `extract()`
3. Вывести результат в читаемом виде

---
## Задание 1 — Извлечение контактных данных (★☆☆)

Из текста ниже извлеките структурированную карточку контакта.

**Схема должна содержать:** `full_name`, `email`, `phone` (Optional), `company`, `position`.

```
Привет! Я Марина Соколова, работаю продакт-менеджером в Яндексе.
Пишите мне на marina.sokolova@yandex.ru или звоните: +7 (916) 123-45-67.
```

In [21]:
# === Ваш код ===

text_1 = """Привет! Я Марина Соколова, работаю продакт-менеджером в Яндексе.
Пишите мне на marina.sokolova@yandex.ru или звоните: +7 (916) 123-45-67 Ещё я хоррор писательница , прочитайте мою книгу "Ужасы Яндекса!!"."""

# 1. Определите класс ContactCard(BaseModel): ...
# 2. result = extract(ContactCard, text_1)
# 3. print(result)

class HH(BaseModel):
    """Модель данных для извлечения возвраста и имени человепка"""
    name: str = Field(description="Полное имя человека")
    email: str = Field(description="email")
    company: Optional[str]= Field(default=None,description="company")
    phone:  int = Field(description="phone")
    position:Optional [str] = Field(default=None,description="position")



In [ ]:

pprint(HH.model_json_schema())

{'description': 'Модель данных для извлечения возвраста и имени человепка',
 'properties': {'company': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                            'default': None,
                            'description': 'company',
                            'title': 'Company'},
                'email': {'description': 'email',
                          'title': 'Email',
                          'type': 'string'},
                'name': {'description': 'Полное имя человека',
                         'title': 'Name',
                         'type': 'string'},
                'phone': {'description': 'phone',
                          'title': 'Phone',
                          'type': 'integer'},
                'position': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                             'default': None,
                             'description': 'position',
                             'title': 'Position'}},
 'required': ['name', 'email', 'phone']

In [ ]:
system_prompt = "Ты извлекатель информации по схеме. Следуя схеме, извлеки данные из текста"
user_prompt = "Текст: Меня зовут Алексей Петров, мне 28 лет, работаю инженером в Москве."

completion = client.beta.chat.completions.parse(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format=Person,
)
result = completion.choices[0].message.parsed
print(result)


system_prompt="Ты извлекатель информации по схеме. Следуя схеме, извлеки данные из текста"
user_prompt=text_1
completion=client.beta.chat.completions.parse(model=MODEL_NAME,
  messages=[ {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      response_format=Person )
result=completion.choices[0].message.parsed
print(result)

name='Марина Соколова' email='marina.sokolova@yandex.ru' company='Яндекс' phone=9161234567 position='продакт-менеджер'


---
## Задание 2 — Классификация обращений в поддержку (★★☆)

Вы строите систему маршрутизации тикетов. Модель должна:
- Определить **категорию** обращения: `billing`, `technical`, `account`, `feature_request`, `other`
- Оценить **приоритет**: `low`, `medium`, `high`, `critical`
- Кратко объяснить свой выбор (`reasoning`)
- Предложить краткий ответ клиенту (`suggested_reply`)

Проклассифицируйте все тикеты из списка ниже.

In [ ]:
tickets = [
    "У меня списали деньги дважды за подписку в этом месяце. Верните деньги!",
    "Приложение вылетает при загрузке фото на Android 14.",
    "Было бы здорово добавить тёмную тему.",
    "Не могу войти в аккаунт — пишет 'неверный пароль', хотя я его точно помню.",
    "Сайт полностью не работает, у нас из-за этого стоит продакшен!!!",
]

# === Ваш код ===

# 1. Определите класс TicketClassification(BaseModel): ...
# 2. Пройдитесь по tickets, вызовите extract() для каждого
# 3. Выведите результат в удобном формате

In [27]:
class  TicketClassification(BaseModel):
   
    reasoning: str = Field(description="Краткое объяснение, почему выбран именно такой sentiment и importance . Пиши на русском")
    sentiment: Literal["billing", "technical", "account", "feature_request"] = Field(
        description="категория обращения"
    )
    importance: Literal["low", "medium", "high", "critical"] = Field(
        description="приоритетность обащения "
    )

pprint(TicketClassification.model_json_schema())

def extract(response_format, user_prompt, system_prompt="Ты — полезный ассистент по систему маршрутизации тикетов Классификация обращений в поддержку.предложения не важны у них низкий приоритет"):
    completion = client.beta.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=response_format,
    )
    return completion.choices[0].message.parsed

system_prompt="Классификация обращений в поддержку , отдавай приоритет важности ошибкам или проблемам . предложения не важны у них низкий приоритет"

tickets = [
    "У меня списали деньги дважды за подписку в этом месяце. Верните деньги!",
    "Приложение вылетает при загрузке фото на Android 14.",
    "Было бы здорово добавить тёмную тему.",
    "Не могу войти в аккаунт — пишет 'неверный пароль', хотя я его точно помню.",
    "Сайт полностью не работает, у нас из-за этого стоит продакшен!!!",
]

for ticket in tickets:
    r = extract(TicketClassification, ticket)
    print(f"[{r.sentiment} | {r.importance}]  {ticket}")
    print(f"  -> reasoning: {r.reasoning}\n")
    

{'properties': {'importance': {'description': 'приоритетность обащения ',
                               'enum': ['low', 'medium', 'high', 'critical'],
                               'title': 'Importance',
                               'type': 'string'},
                'reasoning': {'description': 'Краткое объяснение, почему '
                                             'выбран именно такой sentiment и '
                                             'importance . Пиши на русском',
                              'title': 'Reasoning',
                              'type': 'string'},
                'sentiment': {'description': 'категория обращения',
                              'enum': ['billing',
                                       'technical',
                                       'account',
                                       'feature_request'],
                              'title': 'Sentiment',
                              'type': 'string'}},
 'required': ['reasoning', 'se

---
## Задание 3 — Анализ вакансии и матчинг с резюме (★★★)

Задача из двух шагов:

**Шаг A.** Извлеките из текста вакансии структуру:
- `title` — название позиции
- `company` — компания
- `required_skills` — список обязательных навыков (`list[str]`)
- `nice_to_have_skills` — желательные навыки (`list[str]`)
- `experience_years_min` — минимальный опыт (int)
- `remote` — удалённая работа? (`Literal["yes", "no", "hybrid"]`)

**Шаг B.** Передайте в модель JSON вакансии + текст резюме кандидата.  
Модель должна вернуть:
- `match_score` — оценка соответствия от 0 до 100
- `matched_skills` — какие навыки совпали
- `missing_skills` — каких не хватает
- `reasoning` — обоснование оценки
- `verdict` — `Literal["strong_match", "partial_match", "weak_match"]`

In [ ]:
job_posting = """
Компания "DataFlow" ищет Middle Python-разработчика.
Обязательно: Python 3+, FastAPI, PostgreSQL, Docker, Git.
Будет плюсом: Kubernetes, CI/CD, Redis, опыт с ML-пайплайнами.
Опыт от 3 лет. Гибридный формат (2 дня офис, 3 дня удалённо).
"""

candidate_cv = """
Иван Кузнецов, Python-разработчик, 4 года опыта.
Стек: Python, Django, FastAPI, PostgreSQL, Docker, Git, Linux.
Работал с Redis и Celery. Базово знаком с Kubernetes.
Прошёл курс по ML, но коммерческого опыта с ML-пайплайнами нет.
"""

# === Ваш код ===

# Шаг A:
# 1. Определите класс JobPosting(BaseModel): ...
# 2. job = extract(JobPosting, job_posting)
# 3. print(job)

# Шаг B:
# 1. Определите класс MatchResult(BaseModel): ...
# 2. Сформируйте промпт, в котором передаёте JSON вакансии + текст резюме
#    prompt = f"Вакансия:\n{job.model_dump_json(indent=2)}\n\nРезюме:\n{candidate_cv}"
# 3. match = extract(MatchResult, prompt, system_prompt="Ты — HR-аналитик...")
# 4. Выведите результат

In [ ]:

job_posting = """
Компания "DataFlow" ищет Middle Python-разработчика.
Обязательно: Python 3+, FastAPI, PostgreSQL, Docker, Git.
Будет плюсом: Kubernetes, CI/CD, Redis, опыт с ML-пайплайнами.
Опыт от 3 лет. Гибридный формат (2 дня офис, 3 дня удалённо).
"""



class JobPosting(BaseModel):
    """Модель данных для извлечения """
    title: str = Field(description="название позиции")
    required_skills: list[str] = Field(description="required_skills")
    company: Optional[str]= Field(default=None,description="company")
    nice_to_have_skills: list[str] = Field(description="желательные навыки")
    experience_years_min:Optional [int] = Field(default=None,description="минимальный опыт")
    remote:Literal["yes", "no", "hybrid"]=Field(description="удалённая работа")

pprint(JobPosting.model_json_schema())

system_prompt="Ты извлекатель информации по схеме. Следуя схеме, извлеки данные из текста"
user_prompt=job_posting
completion=client.beta.chat.completions.parse(model=MODEL_NAME,
  messages=[ {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      response_format=JobPosting )
result=completion.choices[0].message.parsed
print(result)
job = extract(JobPosting, job_posting)
print(job)

{'description': 'Модель данных для извлечения ',
 'properties': {'company': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                            'default': None,
                            'description': 'company',
                            'title': 'Company'},
                'experience_years_min': {'anyOf': [{'type': 'integer'},
                                                   {'type': 'null'}],
                                         'default': None,
                                         'description': 'минимальный опыт',
                                         'title': 'Experience Years Min'},
                'nice_to_have_skills': {'description': 'желательные навыки',
                                        'items': {'type': 'string'},
                                        'title': 'Nice To Have Skills',
                                        'type': 'array'},
                'remote': {'description': 'удалённая работа',
                           'enum': ['

In [48]:
candidate_cv = """
Иван Кузнецов, Python-разработчик, 4 года опыта.
Стек: Python, Django, FastAPI, PostgreSQL, Docker, Git, Linux.
Работал с Redis и Celery. Базово знаком с Kubernetes.
Прошёл курс по ML, но коммерческого опыта с ML-пайплайнами нет.
"""

class MatchResult(BaseModel):
    """Результат сопоставления резюме кандидата с вакансией"""
    reasoning: str = Field(description="Обоснование выставленной оценки по всем категориям ")

    
    matched_skills: list[str] = Field(description="Навыки кандидата, совпадающие с требованиями вакансии")
    missing_skills: list[str] = Field(description="Требуемые навыки, которых не хватает у кандидата")
    match_score: int = Field(description="Оценка соответствия кандидата вакансии от 0-100%")
    verdict: Literal["strong_match", "partial_match", "weak_match"] = Field(
        description="Итоговый вердикт по соответствию"
    )


prompt = f"Вакансия:\n{job.model_dump_json(indent=2)}\n\nРезюме:\n{candidate_cv}"
print(repr(prompt))
match = extract(
    MatchResult,
    prompt,
    system_prompt="Ты — HR-аналитик. Оцени соответствие резюме вакансии строго на основе предоставленных данных, без домыслов.")

print(f"Вердикт: {match.verdict}")
print(f"Оценка соответствия: {match.match_score}/100")
print(f"Совпадающие навыки: {', '.join(match.matched_skills)}")
print(f"Не хватает: {', '.join(match.missing_skills)}")
print(f"\nОбоснование: {match.reasoning}")

'Вакансия:\n{\n  "title": "Middle Python Developer",\n  "required_skills": [\n    "Python 3+",\n    "FastAPI",\n    "PostgreSQL",\n    "Docker",\n    "Git"\n  ],\n  "company": "DataFlow",\n  "nice_to_have_skills": [\n    "Kubernetes",\n    "CI/CD",\n    "Redis",\n    "Experience with ML Pipelines"\n  ],\n  "experience_years_min": 3,\n  "remote": "hybrid"\n}\n\nРезюме:\n\nИван Кузнецов, Python-разработчик, 4 года опыта.\nСтек: Python, Django, FastAPI, PostgreSQL, Docker, Git, Linux.\nРаботал с Redis и Celery. Базово знаком с Kubernetes.\nПрошёл курс по ML, но коммерческого опыта с ML-пайплайнами нет.\n'
Вердикт: partial_match
Оценка соответствия: 4/100
Совпадающие навыки: Python 3+, FastAPI, PostgreSQL, Docker, Git
Не хватает: Kubernetes (basic)

Обоснование: Иван Кузнецов соответствует требованиям вакансии в части основных навыков: Python (3+), FastAPI, PostgreSQL, Docker и Git. Наличие опыта работы с Redis и Celery является плюсом. Однако, резюме не соответствует требованию о минималь

---

### Подсказки

- Поле `reasoning` лучше ставить **до** полей с ответом — тогда модель сначала «думает», а потом выдаёт результат.  
- `Field(description=...)` — не обязателен, но заметно улучшает качество.  
- `Optional[str]` — если поле может отсутствовать в тексте.